# Лабораторная 1. Языковая модель из счётчиков

In [ ]:
from collections import Counter
from pathlib import Path
import re
import urllib.request

In [ ]:
url = 'https://raw.githubusercontent.com/nevmenandr/word2vec-russian-novels/751b8e17d82a9765940a778f7a8c0d1086d8a9da/books_before/WarAndPeace.txt'
corpus_path = Path(".cache/WarAndPeace.txt")
corpus_path.parent.mkdir(exist_ok=True)

if not corpus_path.exists():
    urllib.request.urlretrieve(url, corpus_path)

text = corpus_path.read_text(encoding="utf-8-sig")
# Убираю строку с названием книги перед началом текста.
text = text.split("Л. Н. Толстой «Война и мир»", 1)[1].lower()

In [ ]:
word_pattern = re.compile(r"[а-яё]+(?:-[а-яё]+)*")

def tokenize(text):
    return word_pattern.findall(text.lower())

words = tokenize(text)
frequencies = Counter(words)

print("Слов:", len(words))
print("Разных слов:", len(frequencies))
assert len(words) >= 100_000

print("\n20 самых частых слов:")
for word, count in frequencies.most_common(20):
    print(f"{word:15} {count:6}")

Слов в корпусе: 433076
Разных слов: 48131

20 самых частых слов:
и                21084
в                10870
не                8520
что               7600
он                7395
на                6665
с                 5861
как               4011
его               3895
к                 3428
я                 2854
она               2732
но                2725
все               2578
было              2501
это               2434
то                2261
сказал            2000
так               1954
о                 1832


In [ ]:
split = int(len(words) * 0.9)
train = words[:split]
test = words[split:]

# Сохраняю исходный текст, чтобы потом взять из него предложение.
test_start = list(word_pattern.finditer(text))[split].start()
test_text = text[test_start:]
assert tokenize(test_text) == test

print("Обучение:", len(train))
print("Проверка:", len(test))

Обучение: 389768
Проверка: 43308


In [ ]:
class BigramModel:
    def fit(self, words):
        self.word_counts = Counter(words)
        self.bigrams = defaultdict(Counter)
        for previous, word in zip(words, words[1:]):
            self.bigrams[previous][word] += 1

        # Последнее слово текста не имеет следующего слова.
        self.totals = {word: sum(row.values())
                       for word, row in self.bigrams.items()}
        return self

    def prob(self, previous, word):
        count = self.bigrams.get(previous, {}).get(word, 0)
        total = self.totals.get(previous, 0)
        return count / total if total else 0.0

In [ ]:
small = BigramModel().fit(tokenize("я люблю чай я люблю кофе"))
print("P(люблю | я) =", small.prob("я", "люблю"))
print("P(чай | люблю) =", small.prob("люблю", "чай"))

assert small.prob("я", "люблю") == 1
assert small.prob("люблю", "чай") == 0.5
assert small.word_counts["кофе"] == 1
assert small.totals.get("кофе", 0) == 0

P(люблю | я) = 1.0
P(чай | люблю) = 0.5


In [ ]:
model = BigramModel().fit(train)
print("Разных слов в обучении:", len(model.word_counts))
print("Разных пар:", sum(len(row) for row in model.bigrams.values()))

# Для этой таблицы и генерации смотрю обычные частоты без сглаживания.

print("\nЧто чаще всего идёт после слова 'князь':")
for word, count in model.bigrams["князь"].most_common(10):
    print(f"{word:15} {count:5}  P = {model.prob('князь', word):.4f}")

Разных слов в обучении: 45323
Разных пар: 236926

Что чаще всего идёт после слова 'князь':
андрей            775  P = 0.5867
василий           147  P = 0.1113
багратион          36  P = 0.0273
николай            32  P = 0.0242
ипполит            23  P = 0.0174
и                  17  P = 0.0129
не                 13  P = 0.0098
в                   8  P = 0.0061
был                 8  P = 0.0061
болконский          7  P = 0.0053
